# Chapter 11 — Neural Networks That Understand Language: Word Embeddings

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 11:** Intro to Word Embeddings: Neural Networks that Understand Language

## Isi Chapter 11

1. NLP dan contoh tugas klasifikasinya
2. Dari teks ke angka: one-hot encoding dan bag of words
3. Dataset review film IMDB (`reviews.txt` dan `labels.txt`)
4. Embedding layer: memilih baris bobot alih-alih perkalian matriks
5. Membandingkan word embedding dengan jarak Euclidean
6. Fill-in-the-blank: makna ditentukan oleh loss
7. Analogi kata: king - man + woman ≈ queen

## Catatan data

Notebook ini memuat dua file dari repository resmi buku: `reviews.txt` (25.000 review)
dan `labels.txt` (positive/negative). Sel di bawah mengunduhnya otomatis jika file
belum ada. Jalankan di Google Colab.

Perhatian: bagian 5 (fill-in-the-blank) berjalan lama di Python murni, sekitar beberapa
menit. Itu normal.

## 1. NLP dan supervised NLP

NLP (natural language processing) adalah bidang yang berfokus pada pemahaman bahasa
manusia oleh mesin. Banyak tugasnya bisa dibaca sebagai klasifikasi: memberi label pada
bagian teks (misalnya sentimen), menghubungkan dua bagian teks (misalnya coreference),
atau mengisi kata yang hilang.

Neural network hanya memetakan angka ke angka. Jadi langkah pertama adalah mengubah teks
menjadi matriks angka. Cara mengubahnya penting, karena neural network mencari korelasi
antara input dan output. Input yang representasinya tepat membuat korelasi itu lebih
mudah ditemukan.

## 2. Dari kata ke vektor: one-hot encoding

Jika vocabulary terdiri dari empat kata, setiap kata dipetakan ke vektor one-hot. Kalimat
dibuat dengan menjumlahkan vektor setiap kata. Saya cek contoh `the cat sat`:

In [1]:
import numpy as np

onehots = {}
onehots['cat'] = np.array([1, 0, 0, 0])
onehots['the'] = np.array([0, 1, 0, 0])
onehots['dog'] = np.array([0, 0, 1, 0])
onehots['sat'] = np.array([0, 0, 0, 1])

sentence = ['the', 'cat', 'sat']
x = onehots[sentence[0]] + onehots[sentence[1]] + onehots[sentence[2]]
print('Sent Encoding:', x)

Sent Encoding: [1 1 0 1]


Hasilnya `[1 1 0 1]`. Jika sebuah kata muncul berulang, misalnya `cat cat cat`, saya
memakai satu kali saja (bukan dijumlahkan) karena itu biasanya lebih baik untuk bahasa.

## 3. Memuat data review film

Setiap baris `reviews.txt` adalah satu review, dan `labels.txt` berisi sentimennya.

In [2]:
import os, urllib.request

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/{}/{}'

def ensure_file(name):
    if os.path.exists(name):
        return
    for branch in ['master', 'main']:          # cabang repo bisa master atau main
        try:
            urllib.request.urlretrieve(RAW.format(branch, name), name)
            return
        except Exception:
            pass
    raise RuntimeError('Gagal mengunduh ' + name)

ensure_file('reviews.txt')
ensure_file('labels.txt')

f = open('reviews.txt'); raw_reviews = f.readlines(); f.close()
f = open('labels.txt'); raw_labels = f.readlines(); f.close()

print('jumlah review:', len(raw_reviews))
print('jumlah label :', len(raw_labels))
print('contoh label pertama:', raw_labels[0].strip())

jumlah review: 25000
jumlah label : 25000
contoh label pertama: positive


Berikutnya saya membuat vocabulary dari seluruh review, lalu mengubah setiap review
menjadi daftar indeks kata unik. Label `positive` menjadi 1 dan lainnya menjadi 0.

In [3]:
tokens = list(map(lambda x: set(x.split(" ")), raw_reviews))

vocab = set()
for sent in tokens:
    for word in sent:
        if len(word) > 0:
            vocab.add(word)
vocab = sorted(vocab)   # diurutkan agar indeks kata sama setiap kali dijalankan

word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

target_dataset = list()
for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

print('ukuran vocabulary :', len(vocab))
print('contoh input (5 indeks pertama review 0):', input_dataset[0][:5])
print('jumlah target 1   :', sum(target_dataset), 'dari', len(target_dataset))

ukuran vocabulary : 74074
contoh input (5 indeks pertama review 0): [0, 1, 2, 62982, 62983]
jumlah target 1   : 12500 dari 25000


## 4. Embedding layer

Lapisan pertama adalah perkalian matriks antara vektor one-hot dan `weights_0_1`. Karena
vektor input hanya berisi 0 dan 1, hasil perkalian itu sama dengan menjumlahkan beberapa
baris `weights_0_1`. Jadi saya cukup memilih baris untuk setiap kata, lalu menjumlahkannya.
Hasilnya sama, tetapi jauh lebih cepat, karena vocabulary berukuran puluhan ribu kata.

Saya melatih jaringan dengan embedding + sigmoid, lalu sigmoid lagi di output. Latih
pada 24.000 review pertama, lalu uji pada 1.000 review terakhir:

In [4]:
import sys, numpy as np
np.random.seed(1)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100
weights_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

for iter in range(iterations):
    correct, total = (0, 0)
    # Latih pada 24.000 review pertama
    for i in range(len(input_dataset) - 1000):
        x, y = (input_dataset[i], target_dataset[i])

        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))   # embedding + sigmoid
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))    # linear + sigmoid

        layer_2_delta = layer_2 - y
        layer_1_delta = layer_2_delta.dot(weights_1_2.T)

        weights_0_1[x] -= layer_1_delta * alpha
        weights_1_2 -= np.outer(layer_1, layer_2_delta) * alpha

        if np.abs(layer_2_delta) < 0.5:
            correct += 1
        total += 1
    print('Iter:', iter, '| Training Accuracy:', round(correct / float(total), 4))

# Uji pada 1.000 review terakhir
correct, total = (0, 0)
for i in range(len(input_dataset) - 1000, len(input_dataset)):
    x = input_dataset[i]
    y = target_dataset[i]
    layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
    layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
    if np.abs(layer_2 - y) < 0.5:
        correct += 1
    total += 1
print('Test Accuracy:', round(correct / float(total), 4))

Iter: 0 | Training Accuracy: 0.8351
Iter: 1 | Training Accuracy: 0.9012
Test Accuracy: 0.847


Yang saya perhatikan dari hasil ini: akurasi training naik dari iterasi pertama ke
kedua, dan akurasi uji sekitar 0.85. Jaringan ini belajar korelasi antara kata dan label
sentimen dengan cukup baik.

## 5. Membandingkan word embedding

Setiap baris `weights_0_1` adalah embedding sebuah kata. Dua kata dianggap mirip jika
barisnya dekat satu sama lain. Saya memakai jarak Euclidean, dengan nilai negatif supaya
nilai yang lebih besar berarti lebih mirip.

In [5]:
from collections import Counter
import math

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - weights_0_1[target_index]
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)

print(similar('beautiful'))
print(similar('terrible'))

[('beautiful', -0.0), ('fun', -0.7193539861838948), ('friendship', -0.7348320815440881), ('surprisingly', -0.7384555735078454), ('captures', -0.7390758736393925), ('liked', -0.749805555426536), ('subtle', -0.7516892723665487), ('heart', -0.7554179574380616), ('best', -0.7576520996035888), ('different', -0.7579501223137702)]
[('terrible', -0.0), ('annoying', -0.8221713335426316), ('laughable', -0.8304229176743572), ('mess', -0.8324647775644285), ('lacks', -0.8326983682672952), ('effort', -0.8487362920639149), ('save', -0.8493719008649456), ('lame', -0.8503989569601874), ('disappointing', -0.8627308187320908), ('boring', -0.8670990063145886)]


Kata yang paling mirip dengan `beautiful` (selain dirinya sendiri) adalah kata-kata
yang juga punya arah sentimen positif, seperti `fun`, `friendship`, atau `liked`. Untuk
`terrible`, hasilnya kata-kata negatif seperti `annoying`, `laughable`, dan `boring`. Ini wajar: jaringan
hanya belajar membedakan positif dan negatif, jadi kata dikelompokkan menurut label yang
diprediksinya, bukan menurut makna dalam kehidupan nyata.

## 6. Fill-in-the-blank: makna ditentukan oleh loss

Sekarang saya mengubah tugasnya. Dari setiap frasa, satu kata dihapus, dan jaringan
harus menebak kata itu dari kata-kata di sekitarnya. Ini disebut skip-gram dengan
negative sampling: alih-alih memprediksi semua kata (ribuan), jaringan hanya diuji
terhadap kata yang benar dan beberapa kata acak.

Preprocessing berbeda dari bagian sebelumnya: kata tidak dibuang duplikatnya, dan
urutan kata dipertahankan.

Catatan: vocabulary diurutkan (`sorted`). Tanpa pengurutan, urutan kata dari `set` bisa berubah di setiap sesi Python, sehingga bobot awal jatuh pada kata yang berbeda dan hasilnya tidak bisa diulang.

In [6]:
import sys, random, math
from collections import Counter
import numpy as np

np.random.seed(1)
random.seed(1)

tokens = list(map(lambda x: (x.split(" ")), raw_reviews))

wordcnt = Counter()
for sent in tokens:
    for word in sent:
        wordcnt[word] -= 1
vocab = sorted(set(map(lambda x: x[0], wordcnt.most_common())))   # diurutkan agar hasil konsisten

word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

concatenated = list()
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""
    input_dataset.append(sent_indices)
concatenated = np.array(concatenated)
random.shuffle(input_dataset)

print('ukuran vocabulary :', len(vocab))
print('jumlah token total:', len(concatenated))

ukuran vocabulary : 74075
jumlah token total: 7459318


Pelatihannya: untuk setiap kata target, konteks adalah kata di kiri dan kanannya
(dalam `window = 2`). Target benar diberi label 1, dan 5 kata acak (`negative = 5`)
diberi label 0. Hanya kata-kata itulah yang bobot output-nya diperbarui.

Catatan dari notebook resmi: batas konteks kanan dalam kode buku adalah
`target_i+window` (tanpa `+1`), sehingga kata kanan terjauh tidak ikut. Saya tetap
memakai kode buku apa adanya agar hasilnya bisa dibandingkan.

In [7]:
alpha, iterations = (0.05, 2)
hidden_size, window, negative = (50, 2, 5)

weights_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab), hidden_size) * 0

layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - weights_0_1[target_index]
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

for rev_i, review in enumerate(input_dataset * iterations):
    for target_i in range(len(review)):
        # Karena terlalu mahal memprediksi seluruh vocabulary, hanya subset acak yang diprediksi
        target_samples = [review[target_i]] + list(concatenated[(np.random.rand(negative) * len(concatenated)).astype('int').tolist()])

        left_context = review[max(0, target_i - window):target_i]
        right_context = review[target_i + 1:min(len(review), target_i + window)]

        layer_1 = np.mean(weights_0_1[left_context + right_context], axis=0)
        layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))
        layer_2_delta = layer_2 - layer_2_target
        layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])

        weights_0_1[left_context + right_context] -= layer_1_delta * alpha
        weights_1_2[target_samples] -= np.outer(layer_2_delta, layer_1) * alpha

    if rev_i % 5000 == 0:
        print('progress review:', rev_i, '/', len(input_dataset) * iterations)

print(similar('terrible'))

progress review: 0 / 50000
progress review: 5000 / 50000
progress review: 10000 / 50000
progress review: 15000 / 50000
progress review: 20000 / 50000
progress review: 25000 / 50000
progress review: 30000 / 50000
progress review: 35000 / 50000
progress review: 40000 / 50000
progress review: 45000 / 50000
[('terrible', -0.0), ('horrible', -2.9151087306292993), ('brilliant', -3.4279948320918208), ('phenomenal', -3.6613748555764696), ('superb', -3.880567922570928), ('pathetic', -3.8914864275991325), ('masterful', -3.9548880951045127), ('marvelous', -4.008924209789266), ('bad', -4.0608950122945915), ('great', -4.124045272466687)]


Dibandingkan dengan bagian 5, kata-kata yang mirip dengan `terrible` sekarang lebih
beragam. Kata seperti `horrible`, `pathetic`, dan `phenomenal` (yang bisa muncul di
kalimat yang sama) menjadi dekat, walaupun sentimennya tidak selalu sama. Pelajaran
yang saya tangkap: perubahan loss (dari memprediksi positif/negatif menjadi mengisi kata
kosong) mengubah makna yang dipelajari jaringan, walaupun data dan arsitekturnya mirip.

Saya juga memeriksa `beautiful` dengan embedding baru:

In [8]:
print(similar('beautiful'))

[('beautiful', -0.0), ('lovely', -3.268580858914296), ('creepy', -3.2702013664053196), ('nightmarish', -3.5037897633240447), ('spooky', -3.5087611549737767), ('glamorous', -3.5150291282909376), ('gorgeous', -3.523947061463086), ('fantastic', -3.5428291191990278), ('heartwarming', -3.570337263576464), ('fiery', -3.653824304573429)]


## 7. Analogi kata

Embedding hasil fill-in-the-blank bisa dipakai untuk operasi aljabar. Kalau vektor
`king` dikurangi `man` lalu ditambah `woman`, vektor hasilnya seharusnya dekat dengan
`queen`. Fungsi di bawah menormalkan setiap vektor, lalu mencari kata yang paling dekat
dengan hasil operasi itu.

In [9]:
def analogy(positive=['terrible', 'good'], negative=['bad']):
    norms = np.sum(weights_0_1 * weights_0_1, axis=1)
    norms.resize(norms.shape[0], 1)
    normed_weights = weights_0_1 * norms

    query_vect = np.zeros(len(weights_0_1[0]))
    for word in positive:
        query_vect += normed_weights[word2index[word]]
    for word in negative:
        query_vect -= normed_weights[word2index[word]]

    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - query_vect
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)[1:]

print('terrible - bad + good:', analogy(['terrible', 'good'], ['bad']))
print('elizabeth - she + he :', analogy(['elizabeth', 'he'], ['she']))

terrible - bad + good: [('superb', -210.13712222170625), ('terrific', -210.42714339946275), ('decent', -210.55395878237812), ('fine', -210.72072314025576), ('perfect', -210.9312197750544), ('nice', -210.97513790276372), ('brilliant', -211.0444742527515), ('worth', -211.09680959179832), ('terrible', -211.17246750172933)]
elizabeth - she + he : [('christopher', -189.7740476812585), ('it', -190.07210978588074), ('david', -190.15288305130312), ('this', -190.15714262254608), ('him', -190.2620038815329), ('fred', -190.29271681307918), ('mr', -190.31630435382917), ('william', -190.31888177913504), ('bruce', -190.37110041940443)]


Hasil analogi ini bukan jaminan untuk setiap kata. Pola ini muncul karena embedding
merepresentasikan hubungan kata secara linear (misalnya kata yang berhubungan dengan
kerajaan atau gender). Dalam chapter ini, saya mencatat bahwa properti tersebut lebih
berkaitan dengan statistik bahasa daripada dengan deep learning itu sendiri.

## Key Ideas — Chapter 11

- Neural network hanya memetakan angka ke angka, jadi teks harus diubah menjadi vektor.
- One-hot encoding dan bag of words merepresentasikan kehadiran kata dalam sebuah review.
- Embedding layer adalah perkalian one-hot dengan matriks, yang dibuat lebih cepat dengan menjumlahkan baris yang dipilih.
- Kata yang memiliki efek serupa pada prediksi akan memiliki embedding yang mirip.
- Makna sebuah neuron atau kata ditentukan oleh loss: target yang berbeda menghasilkan embedding yang berbeda.
- Analogi kata muncul dari struktur linear dalam statistik co-occurrence.

## Kesimpulan

Chapter 11 menunjukkan bahwa neural network bisa belajar dari teks setelah teks
diubah menjadi representasi angka yang tepat. Word embedding menghasilkan vektor kata
yang bisa dibandingkan dan dioperasikan. Makna dalam jaringan ditentukan oleh target
yang diprediksi, jadi memilih loss adalah keputusan penting. Chapter berikutnya membahas
jaringan rekuren yang memproses urutan kata.